# Gross Price Dimension Ingestion & Transformation Pipeline

This notebook implements an enterprise-grade ETL pipeline using PySpark and Delta Lake on Databricks following the Medallion Architecture pattern (Bronze, Silver, Gold).

## Table of Contents
1. [Environment Setup & Configuration](#1.-Environment-Setup-&-Configuration)
2. [Bronze Layer: Raw Ingestion](#2.-Bronze-Layer:-Raw-Ingestion)
3. [Silver Layer: Cleaning, Normalization & Enrichment](#3.-Silver-Layer:-Cleaning,-Normalization-&-Enrichment)
4. [Gold Layer: Local Dimension Creation & Latest Price Calculation](#4.-Gold-Layer:-Local-Dimension-Creation-&-Latest-Price-Calculation)
5. [Enterprise Integration: Upsert into Parent Dimension](#5.-Enterprise-Integration:-Upsert-into-Parent-Dimension)

---

## 1. Environment Setup & Configuration

Initializes external utility scripts, dynamic Databricks widgets, dynamic parameters, and centralized configuration constants.

In [0]:
%run ../1_setup/utilities

In [0]:
"""
Configuration & Parameter Initialization Module
Sets up path mappings, storage account metadata, catalog references, and widget values.
"""
import pyspark.sql.functions as F
from pyspark.sql import Window
from delta.tables import DeltaTable

# Widgets / Dynamic Parameters
dbutils.widgets.text("catalog", "fmcg")
dbutils.widgets.text("data_source", "gross_price")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

# -------------------------------------------------------------------------
# CENTRALIZED CONFIGURATION
# -------------------------------------------------------------------------
STORAGE_ACCOUNT = "fmcgaccount.dfs.core.windows.net"
CONTAINER_NAME = "sports-bar-dp"
PARENT_PRODUCTS_DIM_TABLE = f"{catalog}.gold.dim_gross_price"

# Static Metadata Default Values
DEFAULT_MARKET = "India"
DEFAULT_PLATFORM = "Sports bar"
DEFAULT_CHANNEL = "Acquisition"

# Derived Data Paths and Table Names
base_path = f"abfss://{CONTAINER_NAME}@{STORAGE_ACCOUNT}/{data_source}/*.csv"
bronze_table_name = f"{catalog}.bronze.{data_source}"
silver_table_name = f"{catalog}.silver.{data_source}"
gold_table_name = f"{catalog}.gold.sb_dim_{data_source}"
products_table_name = f"{catalog}.silver.products"

print(f"Catalog: {catalog}")
print(f"Data Source: {data_source}")
print(f"Base Path: {base_path}")

## 2. Bronze Layer: Raw Ingestion

Loads raw CSV data from Azure Blob Storage/ADLS, appends lineage metadata (ingestion timestamp, filename, file size), and persists the raw records to the Bronze Delta Lake table.

In [0]:
# Load raw CSV data with lineage metadata
try:
    df_raw = (
        spark.read.format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .load(base_path)
        .withColumn("read_timestamp", F.current_timestamp())
        .select("*", F.col("_metadata.file_name").alias("file_name"), F.col("_metadata.file_size").alias("file_size"))
    )
    
    # Exploratory validation
    display(df_raw)
    df_raw.printSchema()
    
    # Write raw ingestion to Bronze Delta table
    (
        df_raw.write
        .format("delta")
        .mode("overwrite")
        .option("delta.enableChangeDataFeed", "true")
        .saveAsTable(bronze_table_name)
    )
    print(f"Successfully written raw data to Bronze table: {bronze_table_name}")
except Exception as e:
    print(f"Error during Bronze ingestion layer processing: {str(e)}")
    raise

## 3. Silver Layer: Cleaning, Normalization & Enrichment

Parses irregular date formats, cleans price amounts, normalizes metadata types, and joins with the product dimension to attach surrogate keys.

In [0]:
# Read from Bronze Delta Table
bronze_df = spark.read.table(bronze_table_name)

# Standard formats to handle multiple incoming date string structures
valid_formats = [
    "yyyy-MM-dd HH:mm:ss",
    "yyyy-MM-dd HH:mm",
    "yyyy-MM-dd",
    "yyyy-MM",
    "yyyy/MM/dd HH:mm:ss",
    "yyyy/MM/dd",
    "yyyy/MM",
    "MM/dd/yyyy HH:mm:ss",
    "MM/dd/yyyy",
    "MM-dd-yyyy",
    "dd/MM/yyyy HH:mm:ss",
    "dd/MM/yyyy",
    "dd-MM-yyyy",
    "yyyyMMdd"
]

# Parse variable string dates into standard timestamps
parsed_timestamp = F.coalesce(*[
    F.try_to_timestamp(F.col("month"), F.lit(fmt))
    for fmt in valid_formats
])

# Standardize attributes and clean gross_price
cleaned_df = (
    bronze_df
    .withColumn("month", F.date_format(parsed_timestamp, "yyyy-MM-dd"))
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn(
        "gross_price",
        F.when(
            F.col("gross_price").rlike("^-?\\d+(\\.\\d+)?$"),
            F.abs(F.col("gross_price").cast("double"))
        ).otherwise(0.0)
    )
)

# Join with product dimension for product_code enrichment
products_table = spark.table(products_table_name)
silver_df = (
    cleaned_df
    .join(products_table.select("product_id", "product_code"), on="product_id", how="inner")
    .select(
        "product_id",
        "product_code",
        "month",
        "gross_price",
        "read_timestamp",
        "file_name",
        "file_size"
    )
)

# Persist to Silver Delta Table
(
    silver_df.write
    .format("delta")
    .option("delta.enableChangeDataFeed", "true")
    .option("mergeSchema", "true")
    .mode("overwrite")
    .saveAsTable(silver_table_name)
)
print(f"Successfully processed and written data to Silver table: {silver_table_name}")

## 4. Gold Layer: Local Dimension Creation & Latest Price Calculation

Extracts refined attributes into a local domain entity, ranks price records per product code and year, and filters to retain only the latest valid non-zero price record per product per year.

In [0]:
# Build Gold base dataset
gold_columns = ["product_code", "gross_price", "month"]
gold_df = silver_df.select(*gold_columns)

# Save domain-specific Gold table
(
    gold_df.write
    .format("delta")
    .option("mergeSchema", "true")
    .mode("overwrite")
    .saveAsTable(gold_table_name)
)

# Feature engineering: derive year, flag missing prices, rank records
gold_df_prepared = (
    gold_df
    .withColumn("year", F.year(F.col("month")))
    .withColumn(
        "is_zero",
        F.when(F.col("gross_price") == 0, 1).otherwise(0)
    )
)

# Window partition to get latest valid price per product per year
window_spec = (
    Window.partitionBy("product_code", "year")
    .orderBy(F.col("is_zero"), F.col("month").desc())
)

df_gold_latest_price = (
    gold_df_prepared
    .withColumn("rank", F.rank().over(window_spec))
    .filter(F.col("rank") == 1)
    .withColumnRenamed("gross_price", "price_inr")
)

display(df_gold_latest_price)

## 5. Enterprise Integration: Upsert into Parent Dimension

Executes dynamic upserts/merges from the calculated local price dimension into the centralized enterprise product dimension.

In [0]:
product_column_mapping = {
    "product_code": "product_code",
    "year": "year",
    "price_inr": "gross_price"
}

try:
    merge_child_to_parent_dim(
        spark=spark,
        child_table=df_gold_latest_price,
        parent_table_name=PARENT_PRODUCTS_DIM_TABLE,
        column_mapping=product_column_mapping,
        merge_key="product_code",
        update_set={
            "year": "source.year",
            "price_inr": "source.price_inr"
        },
        insert_values={
            "product_code": "source.product_code",
            "year": "source.year",
            "price_inr": "source.price_inr"
        }
    )
    print(f"Successfully merged child dimension into parent table: {PARENT_PRODUCTS_DIM_TABLE}")
except Exception as e:
    print(f"Failed merging into parent dimension: {str(e)}")
    raise